In [241]:
import numpy as np
import pandas as pd
# from sklearn.preprocessing import StandardScaler

In [242]:
columns = ['size', 'est_price', 'rooms', 'price']

train_data = [
	[96.0,	900_000,	2,	1_200_000],
	[118.0,	1_250_000,	2,	1_900_000],
	[141.2,	1_500_000,	3,	3_000_000],
	[129.6,	1_450_000,	3,	2_800_000],
	[110.0,	1_150_000,	2,	1_600_000],
	[105.0,	1_000_000,	2,	1_500_000],
]

test_data = [
	[125.0,	1_300_000,	3,	None]
]

In [243]:
train_df = pd.DataFrame(data=train_data, columns=columns)
train_df

,size,est_price,rooms,price
0,96.0,900000,2,1200000
1,118.0,1250000,2,1900000
2,141.2,1500000,3,3000000
3,129.6,1450000,3,2800000
4,110.0,1150000,2,1600000
5,105.0,1000000,2,1500000


In [244]:
test_df = pd.DataFrame(data=test_data, columns=columns)
test_df

,size,est_price,rooms,price
0,125.0,1300000,3,None


In [245]:
DROP_COLS = ['price', 'est_price']

In [246]:
X_train = train_df.drop(columns=DROP_COLS)
X_train

y_train = train_df[['price']]
y_train

,price
0,1200000
1,1900000
2,3000000
3,2800000
4,1600000
5,1500000


In [247]:
X_test = test_df.drop(columns=DROP_COLS)
X_test

y_test = test_df[['price']]
y_test

,price
0,None


In [248]:
# scaler, ysc = StandardScaler(), StandardScaler()
# Xtr = scaler.fit_transform(X_train) # fit on TRAIN only
# Xte = scaler.transform(X_test)

# ytr = ysc.fit_transform(y_train) # scaled y -> used for training
# yte = y_test.to_numpy(float) # raw y -> used for scoring

In [249]:
X_MEAN = X_train.mean()
print(X_MEAN)

X_STD = X_train.std()
print(X_STD)

size     116.633333
rooms      2.333333
dtype: float64
size     16.604778
rooms     0.516398
dtype: float64


In [250]:
Y_MEAN = y_train.mean()
print(Y_MEAN)

Y_STD = y_train.std()
print(Y_STD)

price    2000000.0
dtype: float64
price    734846.922835
dtype: float64


In [251]:
normalized_X_train=(X_train- X_MEAN) / X_STD
normalized_X_train

normalized_X_test=(X_test- X_MEAN) / X_STD
normalized_X_test

,size,rooms
0,0.503871,1.290994


In [252]:
normalized_y_train=(y_train- Y_MEAN) / Y_STD
normalized_y_train

# normalized_y_test=(y_test- Y_MEAN) / Y_STD
# normalized_y_test

,price
0,-1.088662
1,-0.136083
2,1.360828
3,1.088662
4,-0.544331
5,-0.680414


In [253]:
Xtr = normalized_X_train.to_numpy()
ytr = normalized_y_train.to_numpy()

Xte = normalized_X_test.to_numpy()
yte = y_test.to_numpy()

In [254]:
Xtr

array([[-1.24261419, -0.64549722],
       [ 0.08230563, -0.64549722],
       [ 1.4794938 ,  1.29099445],
       [ 0.78089971,  1.29099445],
       [-0.3994834 , -0.64549722],
       [-0.70060154, -0.64549722]])

In [255]:
ytr

array([[-1.08866211],
       [-0.13608276],
       [ 1.36082763],
       [ 1.08866211],
       [-0.54433105],
       [-0.68041382]])

In [256]:
# losses: return (loss, gradient) -> used for training
def mse(y, p):
	return np.mean((p - y) ** 2), 2 * (p - y) / y.size

def softmax_ce(y, logits): # for classification, y is one-hot
	e = np.exp(logits - logits.max(1, keepdims=True)); p = e / e.sum(1, keepdims=True)
	return -np.mean(np.sum(y * np.log(p + 1e-12), 1)), (p - y) / len(y)

# metrics: return a score only -> used for evaluation
def rmse(y, p):
	return np.sqrt(np.mean((y - p) ** 2))

def r2(y, p):
	return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

In [257]:
class Model:
	def __init__(self, n_in, n_hidden, n_out, seed=0):
		rng = np.random.default_rng(seed)
		self.W1 = rng.normal(0, 0.1, (n_in, n_hidden))
		self.b1 = np.zeros(n_hidden)
		self.W2 = rng.normal(0, 0.1, (n_hidden, n_out))
		self.b2 = np.zeros(n_out)

	def forward(self, X):
		self.X = X
		self.z1 = X @ self.W1 + self.b1
		self.a1 = np.maximum(0, self.z1) # ReLU
		return self.a1 @ self.W2 + self.b2

	predict = forward

	def backward(self, d): # d = dLoss/dOutput
		self.dW2 = self.a1.T @ d
		self.db2 = d.sum(0)
		d1 = (d @ self.W2.T) * (self.z1 > 0) # back through W2 and ReLU
		self.dW1 = self.X.T @ d1
		self.db1 = d1.sum(0)

	def step(self, lr):
		self.W1 -= lr * self.dW1
		self.b1 -= lr * self.db1
		self.W2 -= lr * self.dW2
		self.b2 -= lr * self.db2

In [258]:
model = Model(n_in=Xtr.shape[1], n_hidden=20, n_out=1, seed=42)
lr, epochs = 0.05, 500
history = []

for epoch in range(epochs):
	y_hat = model.forward(Xtr) # 1. forward
	loss, gradient = mse(ytr, y_hat) # 2. loss + gradient
	model.backward(gradient) # 3. backward
	model.step(lr) # 4. update weights
	history.append(loss)
	if epoch % 100 == 0:
		print(f"epoch {epoch:4d}  loss {loss:.4f}")

epoch    0  loss 0.8506
epoch  100  loss 0.0069
epoch  200  loss 0.0008
epoch  300  loss 0.0006
epoch  400  loss 0.0005


In [259]:
# y_pred_train = ysc.inverse_transform(model.predict(Xtr)) # back to original units
# y_pred_test = ysc.inverse_transform(model.predict(Xte))
# y_true_train = ysc.inverse_transform(ytr) # ytr is scaled, unscale it to compare

# print(f"Train  RMSE: {rmse(y_true_train, y_pred_train):.2f}  R2: {r2(y_true_train, y_pred_train):.3f}")
# print(f"Test   RMSE: {rmse(yte, y_pred_test):.2f}  R2: {r2(yte, y_pred_test):.3f}")

In [260]:
tmp1 = [Y_MEAN, Y_MEAN, Y_MEAN, Y_MEAN, Y_MEAN, Y_MEAN]
tmp2 = [Y_STD, Y_STD, Y_STD, Y_STD, Y_STD, Y_STD]

In [261]:
y_pred_train = tmp1 + model.predict(Xtr) * tmp2
y_true_train = tmp1 + ytr * tmp2

print(f"Train  RMSE: {rmse(y_true_train, y_pred_train):.2f}  R2: {r2(y_true_train, y_pred_train):.3f}")

Train  RMSE: 16337.61  R2: 0.999


In [262]:
y_pred_test = [Y_MEAN] + model.predict(Xte) * [Y_STD]
print(y_pred_test)

[[2719149.18332444]]
